#  MODEL TRAINING & VALIDATION — HOUSE PRICES DATASET
### Random Forest vs XGBoost

This notebook is meant to run **side by side** with your feature engineering notebook.

**How to keep them in sync:** at the end of your feature engineering notebook, export the
cleaned/engineered dataframe to CSV, e.g.:

```python
train_fe.to_csv('../data/train_processed.csv', index=False)
test_fe.to_csv('../data/test_processed.csv', index=False)
```

The loader cell below (Phase 0) will automatically use `train_processed.csv` if it exists.
Until it exists, this notebook falls back to a **minimal baseline preprocessing** (basic
NA handling + one-hot encoding) so you can already train and validate models today, and
swap in your real engineered features later without touching anything downstream.

## PHASE 0 — SETUP & DATA LOADING

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, norm, probplot
import warnings
from sklearn.feature_selection import f_regression
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import os
import joblib #used to save model

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:


# CELL : Load Dataset
train_model = pd.read_csv('../data/train_processed.csv')
test = pd.read_csv('../data/test.csv')


print(f"train_modeling data: {train_model.shape[0]} rows × {train_model.shape[1]} columns")
print(f"Test data: {test.shape[0]} rows × {test.shape[1]} columns\n\n")
print("=" * 80)
print("SHAPE")
print("=" * 80)
print(f"Rows: {train_model.shape[0]}, Columns: {train_model.shape[1]}")

print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)
print(train_model.dtypes)


train_modeling data: 1456 rows × 82 columns
Test data: 1459 rows × 80 columns


SHAPE
Rows: 1456, Columns: 82

DATA TYPES
Id                 int64
MSSubClass         int64
MSZoning             str
LotFrontage      float64
LotArea            int64
                  ...   
YrSold             int64
SaleType             str
SaleCondition        str
SalePrice          int64
SalePrice_Log    float64
Length: 82, dtype: object


# BUILD X / y AND SPLIT 

In [ ]:

## =============================================================================
# BUILD X / y AND SPLIT (train_model = your data after outlier treatment)
# =============================================================================

y = train_model['SalePrice_Log']
X = train_model.drop(columns=[c for c in ['SalePrice', 'SalePrice_Log', 'Id'] if c in train_model.columns])


#  ENCODING: Ordinal (preserve order) vs Nominal (one-hot)

In [ ]:

# ORDINAL COLUMNS (have natural order)
ordinal_cols = ['OverallQual', 'OverallCond', 'ExterQual', 'ExterCond', 
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageFinish', 
                'GarageQual', 'GarageCond', 'PavedDrive', 'Functional', 
                'LotShape', 'Utilities', 'LandSlope', 'Fence', 'PoolQC']

# NOMINAL COLUMNS (no order - use get_dummies)
nominal_cols = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotConfig', 
                'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 
                'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 
                'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 
                'MiscFeature', 'SaleType', 'SaleCondition', 'LandContour']

# Step 0: Ensure integer-coded nominal columns (like MSSubClass) are treated as strings
for col in nominal_cols:
    if col in X.columns:
        X[col] = X[col].astype(str)

# Step 1: Fill NA in ordinal columns with 'NA' string, then label encode
for col in ordinal_cols:
    if col in X.columns:
        X[col] = X[col].fillna('NA')
        X[col] = X[col].astype('category').cat.codes  # Converts to 0,1,2... preserving order

# Step 2: One-hot encode nominal columns only
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

# Handle any remaining missing values
X = X.fillna(-1)

print(f"Final feature matrix: {X.shape[1]} columns")

# -------------------------------------------------------------------------
# TRAIN/VAL SPLIT
# -------------------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")
# ====================================================================================================================================


Final feature matrix: 210 columns
X_train: (1164, 210) | X_val: (292, 210)


# ===MODEL 1: RANDOM FOREST===

In [ ]:
print("=" * 60)
print("MODEL 1: RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestRegressor(
    n_estimators=200,       # Build 200 independent trees
    max_depth=6,            # Limit tree depth to prevent overfitting
    min_samples_split=8,
    min_samples_leaf=3,
    random_state=42,        # Lock random seed for reproducibility
    n_jobs=-1                # Use all CPU cores for fast training
)

print("Training...")
rf_model.fit(X_train, y_train)
print("✅ Training complete!")

y_train_pred_rf = rf_model.predict(X_train)


# y_train/y_val are log(SalePrice) -- invert with expm1 to score in real dollars
train_rmse_rf = np.sqrt(mean_squared_error(np.expm1(y_train), np.expm1(y_train_pred_rf)))
train_r2_rf = r2_score(y_train, y_train_pred_rf)

print("\n" + "-" * 60)
print("RANDOM FOREST RESULTS on 80% train")
print("-" * 60)
print(f"Training RMSE: ${train_rmse_rf:,.0f}")
print(f"Training R²: {train_r2_rf:.4f}")


MODEL 1: RANDOM FOREST
Training...
✅ Training complete!

------------------------------------------------------------
RANDOM FOREST RESULTS on 80% train
------------------------------------------------------------
Training RMSE: $19,768
Training R²: 0.9285


In [ ]:
# 1. Generate predictions using the 20% validation features
y_val_pred_rf = rf_model.predict(X_val)

# y_val_pred_rf = rf_model.predict(X_val)
# 2. Calculate RMSE in real dollars (using expm1 to invert the log-transform)
val_rmse_rf = np.sqrt(
    mean_squared_error(np.expm1(y_val), np.expm1(y_val_pred_rf))
)

# 3. Calculate the R2 score on the 20% validation target values
val_r2_rf = r2_score(y_val, y_val_pred_rf)

# val_rmse_rf = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(y_val_pred_rf)))
# val_r2_rf = r2_score(y_val, y_val_pred_rf)

print("\n" + "-" * 60)
print("RANDOM FOREST RESULTS on 20% test")
print("-" * 60)
print(f"Validation RMSE: ${val_rmse_rf:,.0f}")
print(f"Overfit gap (val - train RMSE): ${val_rmse_rf - train_rmse_rf:,.0f}")
print(f"Validation R²: {val_r2_rf:.4f}")



------------------------------------------------------------
RANDOM FOREST RESULTS on 20% test
------------------------------------------------------------
Validation RMSE: $26,724
Overfit gap (val - train RMSE): $6,956
Validation R²: 0.8411


# MODEL 2: XGBOOST

In [ ]:

print("\n" + "=" * 60)
print("MODEL 2: XGBOOST")
print("=" * 60)

xgb_model = xgb.XGBRegressor(
    n_estimators=200,          # 200 sequential boosting rounds
    max_depth=3,               # Shallow trees to avoid memorizing data
    learning_rate=0.025,         # Step size shrinkage for error correction
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training...")
xgb_model.fit(X_train, y_train)
print("✅ Training complete!")

y_train_pred_xgb = xgb_model.predict(X_train)
train_rmse_xgb = np.sqrt(mean_squared_error(np.expm1(y_train), np.expm1(y_train_pred_xgb)))
train_r2_xgb = r2_score(y_train, y_train_pred_xgb)

print("\n" + "-" * 60)
print("XGBOOST RESULTS on 80% train")
print("-" * 60)
print(f"Training RMSE: ${train_rmse_xgb:,.0f}")
print(f"Training R²: {train_r2_xgb:.4f}")

# =============================================================================
# # HEAD-TO-HEAD COMPARISON
# # =============================================================================
# print("\n" + "=" * 60)
# print("MODEL COMPARISON")
# print("=" * 60)


# best_model_name = comparison.loc[comparison['Val RMSE'].idxmin(), 'Model']
# print(f"\n🏆 Best model by validation RMSE: {best_model_name}")


MODEL 2: XGBOOST
Training...
✅ Training complete!

------------------------------------------------------------
XGBOOST RESULTS on 80% train
------------------------------------------------------------
Training RMSE: $18,692
Training R²: 0.9402


In [ ]:

print("\n" + "-" * 60)
print("XGBOOST RESULTS on 20% test")
print("-" * 60)
y_val_pred_xgb = xgb_model.predict(X_val)
val_rmse_xgb = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(y_val_pred_xgb)))
val_r2_xgb = r2_score(y_val, y_val_pred_xgb)
print(f"Validation R²: {val_r2_xgb:.4f}")
print(f"Validation RMSE: ${val_rmse_xgb:,.0f}")
print(f"Overfit gap (val - train RMSE): ${val_rmse_xgb - train_rmse_xgb:,.0f}")


# comparison = pd.DataFrame({
#     'Model': ['Random Forest', 'XGBoost'],
#     'Train RMSE': [train_rmse_rf, train_rmse_xgb],
#     'Val RMSE': [val_rmse_rf, val_rmse_xgb],
#     'Train R²': [train_r2_rf, train_r2_xgb],
#     'Val R²': [val_r2_rf, val_r2_xgb],
# })
# print(comparison.to_string(index=False))


------------------------------------------------------------
XGBOOST RESULTS on 20% test
------------------------------------------------------------
Validation R²: 0.8883
Validation RMSE: $21,899
Overfit gap (val - train RMSE): $3,208


# saving model


In [ ]:
os.makedirs('../trained_model', exist_ok=True)

# 2. Save both models with proper names
rf_path = '../trained_model/random_forest_model_phase3.pkl'
xgb_path = '../trained_model/xgboost_model_phase3.pkl'

joblib.dump(rf_model, rf_path)
joblib.dump(xgb_model, xgb_path)

print('=' * 60)
print('✅ MODELS SUCCESSFULLY SAVED!')
print('=' * 60)
print(f'• Random Forest saved to: {rf_path}')
print(f'• XGBoost saved to: {xgb_path}')

✅ MODELS SUCCESSFULLY SAVED!
• Random Forest saved to: ../trained_model/random_forest_model_phase3.pkl
• XGBoost saved to: ../trained_model/xgboost_model_phase3.pkl
